# Graphe de Synergies
**Objectif :** construire un graphe NetworkX où :
- Chaque **nœud** = une carte
- Chaque **arête** = une synergie mesurée (Jaccard >= 0.3, count >= 5)
- Le **poids** de l'arête = score Jaccard

Ce graphe permet d'identifier les cartes centrales d'un archetype,
les ponts entre archetypes, et l'impact d'un futur ban.

In [ ]:
import sqlite3
import pandas as pd
import networkx as nx

con = sqlite3.connect('../data/yugioh.db')

# Charger les paires significatives
# Filtre : Jaccard >= 0.3 ET au moins 5 decks en commun
pairs = pd.read_sql("""
    SELECT card_a, card_b, jaccard, cooc_count
    FROM card_cooccurrence
    WHERE jaccard >= 0.3
    AND cooc_count >= 5
    ORDER BY jaccard DESC
""", con)

print(f'Paires chargées : {len(pairs):,}')

## 1. Construction du graphe

In [ ]:
G = nx.Graph()

for _, row in pairs.iterrows():
    G.add_edge(row['card_a'], row['card_b'],
               weight=row['jaccard'],
               count=row['cooc_count'])

print(f'Noeuds (cartes) : {G.number_of_nodes():,}')
print(f'Aretes (synergies) : {G.number_of_edges():,}')
print(f'Composantes connexes : {nx.number_connected_components(G):,}')

## 2. Centralite — quelles cartes sont les plus connectees ?

In [ ]:
degree = pd.Series(dict(G.degree(weight='weight')), name='weighted_degree')
degree = degree.sort_values(ascending=False)

print('Top 20 cartes les plus connectees :')
degree.head(20)

## 3. Communautes — detection automatique des archetypes

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G, weight='weight'))
print(f'{len(communities)} communautes detectees')
print()

communities_sorted = sorted(communities, key=len, reverse=True)
for i, comm in enumerate(communities_sorted[:8]):
    cards_list = sorted(comm)[:6]
    print(f'Communaute {i+1} ({len(comm)} cartes) : {", ".join(cards_list)}...')

In [ ]:
## Classification des cartes — format staples vs archetype pieces

import sqlite3, pandas as pd

con_tmp = sqlite3.connect('../data/yugioh.db')

# ── Fréquence globale (% decks légaux contenant la carte, main deck) ──────────
total_decks = pd.read_sql(
    "SELECT COUNT(*) as n FROM tournament_decks WHERE illegal = 0", con_tmp
).iloc[0]['n']

freq_df = pd.read_sql("""
    SELECT dc.card_name,
           COUNT(DISTINCT dc.deck_id) AS deck_count
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0 AND dc.zone = 'main'
    GROUP BY dc.card_name
""", con_tmp)
freq_df['frequency'] = freq_df['deck_count'] / total_decks

# ── Dispersion cross-archetype : dans combien d'archetypes distincts ──────────
# (calculé sur les données brutes, pas le graphe)
arch_df = pd.read_sql("""
    SELECT dc.card_name, td.archetype
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0 AND dc.zone = 'main' AND td.archetype IS NOT NULL
""", con_tmp)
con_tmp.close()

arch_spread = (arch_df.groupby('card_name')['archetype']
               .nunique()
               .reset_index()
               .rename(columns={'archetype': 'n_archetypes'}))

# ── Fusion & classification ───────────────────────────────────────────────────
card_class = freq_df.merge(arch_spread, on='card_name', how='left')
card_class['n_archetypes'] = card_class['n_archetypes'].fillna(1).astype(int)
card_class['frequency_pct'] = (card_class['frequency'] * 100).round(1)

# Seuils :
#   frequency > 25%  → carte jouée dans beaucoup de decks
#   n_archetypes > 5 → carte présente dans plusieurs archetypes distincts
def classify(row):
    hi_freq  = row['frequency'] > 0.25
    hi_cross = row['n_archetypes'] > 5
    if hi_freq and hi_cross:     return 'staple_format'
    if hi_freq and not hi_cross: return 'staple_archetype'
    if not hi_freq and hi_cross: return 'tech_pont'
    return 'piece_niche'

card_class['card_type'] = card_class.apply(classify, axis=1)

print('Distribution des types :')
print(card_class['card_type'].value_counts().to_string())
print()

print('=== STAPLES DE FORMAT (présentes dans beaucoup de decks ET archetypes) ===')
sf = card_class[card_class['card_type'] == 'staple_format'].sort_values('frequency', ascending=False)
print(sf[['card_name', 'frequency_pct', 'n_archetypes']].head(25).to_string(index=False))
print()

print('=== CARTES PONT (tech présente dans plusieurs archetypes, peu fréquente) ===')
tp = card_class[card_class['card_type'] == 'tech_pont'].sort_values('n_archetypes', ascending=False)
print(tp[['card_name', 'frequency_pct', 'n_archetypes']].head(20).to_string(index=False))
print()

print('=== STAPLES ARCHETYPE (dominant dans un seul archetype fort) ===')
sa = card_class[card_class['card_type'] == 'staple_archetype'].sort_values('frequency', ascending=False)
print(sa[['card_name', 'frequency_pct', 'n_archetypes']].head(20).to_string(index=False))

## 4. Focus sur un archetype — sous-graphe Maliss

In [ ]:
def archetype_subgraph(G, keyword, min_jaccard=0.3):
    nodes = [n for n in G.nodes() if keyword.lower() in n.lower()]
    neighbors = set(nodes)
    for n in nodes:
        for neighbor, data in G[n].items():
            if data['weight'] >= min_jaccard:
                neighbors.add(neighbor)
    return G.subgraph(neighbors)

maliss_sg = archetype_subgraph(G, 'Maliss')
print(f'Sous-graphe Maliss : {maliss_sg.number_of_nodes()} cartes, {maliss_sg.number_of_edges()} synergies')
print()

maliss_degree = pd.Series(dict(maliss_sg.degree(weight='weight'))).sort_values(ascending=False)
print('Cartes les plus centrales dans Maliss :')
maliss_degree.head(10)

## 5. Simulation de ban — impact sur le graphe

In [ ]:
def simulate_ban(G, card_name):
    if card_name not in G:
        print(f'Carte "{card_name}" introuvable dans le graphe.')
        return
    
    neighbors = list(G.neighbors(card_name))
    weights = [G[card_name][n]['weight'] for n in neighbors]
    impact = pd.Series(weights, index=neighbors).sort_values(ascending=False)
    
    print(f'Ban de "{card_name}"')
    print(f'  Connexions supprimees : {len(neighbors)}')
    print(f'  Cartes les plus impactees :')
    for card, w in impact.head(10).items():
        print(f'    {w:.2f}  {card}')

simulate_ban(G, 'Ash Blossom & Joyous Spring')

In [ ]:
simulate_ban(G, 'Maliss P March Hare')

## 6. Visualisation interactive — Pyvis

In [ ]:
from pyvis.network import Network

def visualize_archetype(G, keyword, min_jaccard=0.3, output_file=None):
    """
    Génère un graphe interactif HTML pour un archetype.
    - Taille du noeud = centralité
    - Épaisseur de l'arête = score Jaccard
    - Rouge = cartes core de l'archetype, Bleu = cartes externes
    """
    sg = archetype_subgraph(G, keyword, min_jaccard)
    core_cards = set(n for n in sg.nodes() if keyword.lower() in n.lower())

    net = Network(height='700px', width='100%', bgcolor='#1a1a2e', font_color='white')
    net.set_options('''
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -80,
          "springLength": 120
        },
        "solver": "forceAtlas2Based"
      }
    }
    ''')

    degrees = dict(sg.degree(weight='weight'))
    max_deg = max(degrees.values()) if degrees else 1

    for node in sg.nodes():
        size = 15 + 35 * (degrees[node] / max_deg)
        color = '#e94560' if node in core_cards else '#4a90d9'
        net.add_node(node, label=node, size=size, color=color,
                     title=f'Centralité: {degrees[node]:.2f}')

    for u, v, data in sg.edges(data=True):
        net.add_edge(u, v, value=data['weight'],
                     title=f"Jaccard: {data['weight']:.2f} ({data['count']} decks)")

    if output_file is None:
        output_file = f'../data/graph_{keyword.lower().replace(" ", "_")}.html'

    net.save_graph(output_file)
    print(f'✓ {output_file} — {sg.number_of_nodes()} cartes, {sg.number_of_edges()} synergies')
    return output_file

# Générer les graphes des top archetypes
for archetype in ['Maliss', 'Tenpai', 'Ryzeal', 'Branded']:
    visualize_archetype(G, archetype)
